# Candidate fitting and model selection

Run this notebook only after the setup/smoke-test stage has placed a pretrained
IResNet-100 checkpoint at the configured local-only path. It performs ten fits:
three Triplet Margin, four ArcFace, and three SupCon. No selected candidate is
retrained on validation data.

In [9]:
%pip install -q pytorch-metric-learning h5py

import importlib
import shutil
import sys
from pathlib import Path

print("hello")

# Configuration
REPOSITORY_ROOT = next(
    root for root in (Path.cwd().resolve(), Path.cwd().resolve().parent)
    if (root / "GLOSSARY.md").exists() and (root / "face_name_metric_learning").exists()
)
MANIFEST_PATH = REPOSITORY_ROOT / "data" / "processed" / "manifest.csv"
SOURCE_HDF5_PATH = REPOSITORY_ROOT / "data" / "processed" / "faces.h5"
PRETRAINED_CHECKPOINT_PATH = (
    REPOSITORY_ROOT / "models" / "pretrained" / "iresnet100.pth"
)
TRIAL_RESULTS_PATH = (
    REPOSITORY_ROOT / "artifacts" / "face_name_metric_learning" / "trial_results.csv"
)
EXPECTED_SPLIT_COUNTS = {"train": 45_078, "validation": 9_660, "test": 9_660}
EXPECTED_CLASS_COUNT = 100

HDF5_PATH = REPOSITORY_ROOT / "data" / "processed" / "faces.h5"

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

import torch
import pytorch_metric_learning
import face_name_metric_learning.pytorch_training as pytorch_training

importlib.reload(pytorch_training)

from face_name_metric_learning import (
    DEFAULT_TRAINING_CONFIG,
    candidate_configurations,
    read_manifest,
    trial_checkpoint_path,
    validate_manifest_hdf5,
    validate_trial_ledger,
)
from face_name_metric_learning.pytorch_training import train_and_select_candidates

candidates = candidate_configurations()
assert len(candidates) == 10
assert [candidate.loss for candidate in candidates].count("triplet_margin") == 3
assert [candidate.loss for candidate in candidates].count("arcface") == 4
assert [candidate.loss for candidate in candidates].count("supcon") == 3
assert torch.cuda.is_available(), "A CUDA device is required; training is not run on CPU."
assert PRETRAINED_CHECKPOINT_PATH.is_file(), (
    f"Pretrained checkpoint missing: {PRETRAINED_CHECKPOINT_PATH}."
)

device = torch.device("cuda")
print({
    "device": str(device),
    "gpu": torch.cuda.get_device_name(device),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "pytorch_metric_learning": pytorch_metric_learning.__version__,
}, flush=True)

data_contract = validate_manifest_hdf5(
    MANIFEST_PATH,
    HDF5_PATH,
    expected_split_counts=EXPECTED_SPLIT_COUNTS,
    expected_class_count=EXPECTED_CLASS_COUNT,
)
manifest_rows = read_manifest(MANIFEST_PATH)
assert data_contract["split_counts"] == EXPECTED_SPLIT_COUNTS
assert data_contract["class_count"] == EXPECTED_CLASS_COUNT
print(data_contract, flush=True)

trial_rows = train_and_select_candidates(
    repository_root=REPOSITORY_ROOT,
    manifest_path=MANIFEST_PATH,
    hdf5_path=HDF5_PATH,
    trial_results_path=TRIAL_RESULTS_PATH,
    pretrained_checkpoint_path=PRETRAINED_CHECKPOINT_PATH,
    manifest_rows=manifest_rows,
    config=DEFAULT_TRAINING_CONFIG,
)

ledger = validate_trial_ledger(TRIAL_RESULTS_PATH)
assert len(ledger["rows"]) == 10
print("Selected trials:", ledger["selected_trials"], flush=True)

display_columns = [
    "trial_id",
    "loss",
    "hyperparameters_json",
    "epochs_attempted",
    "best_epoch",
    "best_validation_triplet_accuracy",
    "notes",
    "selected",
]
selected = [
    {column: row[column] for column in display_columns}
    for row in sorted(
        trial_rows,
        key=lambda row: (row["loss"], -float(row["best_validation_triplet_accuracy"])),
    )
]
for row in selected:
    print(row, flush=True)

checkpoint_paths = [trial_checkpoint_path(REPOSITORY_ROOT, row) for row in trial_rows]
assert all(path.is_file() for path in checkpoint_paths)
assert all("models/metric_learning" in path.as_posix() for path in checkpoint_paths)
print("Best checkpoints:", flush=True)
for path in checkpoint_paths:
    print(path.relative_to(REPOSITORY_ROOT), flush=True)


Note: you may need to restart the kernel to use updated packages.
hello
{'device': 'cuda', 'gpu': 'NVIDIA L4', 'torch': '2.8.0+cu128', 'cuda': '12.8', 'pytorch_metric_learning': '2.9.0'}
{'rows': 64398, 'split_counts': {'train': 45078, 'test': 9660, 'validation': 9660}, 'class_count': 100}
Loading /nfs/home/aqu4eb/DS4002_Project3/data/processed/faces.h5 into RAM (~9.7 GB, one-time)...
Loaded into RAM.
[triplet_margin0p1_seed4002] epoch 1/15 val_acc=0.6466 best=0.6466 improved=True
[triplet_margin0p1_seed4002] epoch 2/15 val_acc=0.6549 best=0.6549 improved=True
[triplet_margin0p1_seed4002] epoch 3/15 val_acc=0.6568 best=0.6568 improved=True
[triplet_margin0p1_seed4002] epoch 4/15 val_acc=0.6573 best=0.6573 improved=True
[triplet_margin0p1_seed4002] epoch 5/15 val_acc=0.6599 best=0.6599 improved=True
[triplet_margin0p1_seed4002] epoch 6/15 val_acc=0.6602 best=0.6602 improved=True
[triplet_margin0p1_seed4002] epoch 7/15 val_acc=0.6631 best=0.6631 improved=True
[triplet_margin0p1_seed4002]

LooseLeaf policy: GPU cells may run for 2 h 00 min; this one is interrupted in 15 min (its state is saved as for any interrupted cell). Split long training into cells that each run less.


[triplet_margin0p4_seed4002] epoch 7/15 val_acc=0.6785 best=0.6794 improved=False
[triplet_margin0p4_seed4002] epoch 8/15 val_acc=0.6813 best=0.6813 improved=True
[triplet_margin0p4_seed4002] epoch 9/15 val_acc=0.6791 best=0.6813 improved=False
[triplet_margin0p4_seed4002] epoch 10/15 val_acc=0.6781 best=0.6813 improved=False
[triplet_margin0p4_seed4002] epoch 11/15 val_acc=0.6743 best=0.6813 improved=False


LooseLeaf policy: this GPU cell reached the 2 h 00 min limit and was interrupted.


KeyboardInterrupt: 

In [8]:
import os, shutil
from pathlib import Path

for p in ("/tmp/faces_uncompressed.h5", "/tmp/faces.h5"):
    Path(p).unlink(missing_ok=True)

mem_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / (1024**3)
print(f"RAM: {mem_gb:.1f} GB")
print(f"/tmp free: {shutil.disk_usage('/tmp').free / (1024**3):.1f} GB")
print(f"NFS free:  {shutil.disk_usage(Path.home()).free / (1024**3):.1f} GB")

RAM: 251.5 GB
/tmp free: 8.0 GB
NFS free:  68.5 GB


In [14]:
REPOSITORY_ROOT = next(
    root for root in (Path.cwd().resolve(), Path.cwd().resolve().parent)
    if (root / "GLOSSARY.md").exists() and (root / "face_name_metric_learning").exists()
)

MANIFEST_PATH = REPOSITORY_ROOT / "data" / "processed" / "manifest.csv"
SOURCE_HDF5_PATH = REPOSITORY_ROOT / "data" / "processed" / "faces.h5"
PRETRAINED_CHECKPOINT_PATH = (
    REPOSITORY_ROOT / "models" / "pretrained" / "iresnet100.pth"
)
TRIAL_RESULTS_PATH = (
    REPOSITORY_ROOT / "artifacts" / "face_name_metric_learning" / "trial_results.csv"
)
EXPECTED_SPLIT_COUNTS = {"train": 45_078, "validation": 9_660, "test": 9_660}
EXPECTED_CLASS_COUNT = 100

HDF5_PATH = REPOSITORY_ROOT / "data" / "processed" / "faces.h5"

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

import importlib
import face_name_metric_learning.pytorch_training as pytorch_training

importlib.reload(pytorch_training)

trial_rows = pytorch_training.train_and_select_candidates(
    repository_root=REPOSITORY_ROOT,
    manifest_path=MANIFEST_PATH,
    hdf5_path=HDF5_PATH,
    trial_results_path=TRIAL_RESULTS_PATH,
    pretrained_checkpoint_path=PRETRAINED_CHECKPOINT_PATH,
    manifest_rows=manifest_rows,
    config=DEFAULT_TRAINING_CONFIG,
    max_new_trials=2,
)

[triplet_margin0p1_seed4002] already in ledger, skipping.
[triplet_margin0p2_seed4002] already in ledger, skipping.
Loading /nfs/home/aqu4eb/DS4002_Project3/data/processed/faces.h5 into RAM (~9.7 GB, one-time)...
Loaded into RAM.
[triplet_margin0p4_seed4002] epoch 1/15 val_acc=0.6697 best=0.6697 improved=True
[triplet_margin0p4_seed4002] epoch 2/15 val_acc=0.6751 best=0.6751 improved=True
[triplet_margin0p4_seed4002] epoch 3/15 val_acc=0.6783 best=0.6783 improved=True
[triplet_margin0p4_seed4002] epoch 4/15 val_acc=0.6779 best=0.6783 improved=False
[triplet_margin0p4_seed4002] epoch 5/15 val_acc=0.6778 best=0.6783 improved=False
[triplet_margin0p4_seed4002] epoch 6/15 val_acc=0.6794 best=0.6794 improved=True
[triplet_margin0p4_seed4002] epoch 7/15 val_acc=0.6785 best=0.6794 improved=False
[triplet_margin0p4_seed4002] epoch 8/15 val_acc=0.6813 best=0.6813 improved=True
[triplet_margin0p4_seed4002] epoch 9/15 val_acc=0.6791 best=0.6813 improved=False
[triplet_margin0p4_seed4002] epoch 10

In [16]:
REPOSITORY_ROOT = next(
    root for root in (Path.cwd().resolve(), Path.cwd().resolve().parent)
    if (root / "GLOSSARY.md").exists() and (root / "face_name_metric_learning").exists()
)

MANIFEST_PATH = REPOSITORY_ROOT / "data" / "processed" / "manifest.csv"
SOURCE_HDF5_PATH = REPOSITORY_ROOT / "data" / "processed" / "faces.h5"
PRETRAINED_CHECKPOINT_PATH = (
    REPOSITORY_ROOT / "models" / "pretrained" / "iresnet100.pth"
)
TRIAL_RESULTS_PATH = (
    REPOSITORY_ROOT / "artifacts" / "face_name_metric_learning" / "trial_results.csv"
)
EXPECTED_SPLIT_COUNTS = {"train": 45_078, "validation": 9_660, "test": 9_660}
EXPECTED_CLASS_COUNT = 100

HDF5_PATH = REPOSITORY_ROOT / "data" / "processed" / "faces.h5"

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

import importlib
import face_name_metric_learning.pytorch_training as pytorch_training

importlib.reload(pytorch_training)

trial_rows = pytorch_training.train_and_select_candidates(
    repository_root=REPOSITORY_ROOT,
    manifest_path=MANIFEST_PATH,
    hdf5_path=HDF5_PATH,
    trial_results_path=TRIAL_RESULTS_PATH,
    pretrained_checkpoint_path=PRETRAINED_CHECKPOINT_PATH,
    manifest_rows=manifest_rows,
    config=DEFAULT_TRAINING_CONFIG,
    max_new_trials=2,
)

[triplet_margin0p1_seed4002] already in ledger, skipping.
[triplet_margin0p2_seed4002] already in ledger, skipping.
[triplet_margin0p4_seed4002] already in ledger, skipping.
[arcface_m0p2_s30_seed4002] already in ledger, skipping.
[arcface_m0p2_s64_seed4002] already in ledger, skipping.
[arcface_m0p4_s30_seed4002] already in ledger, skipping.
Loading /nfs/home/aqu4eb/DS4002_Project3/data/processed/faces.h5 into RAM (~9.7 GB, one-time)...
Loaded into RAM.
[arcface_m0p4_s64_seed4002] epoch 1/15 val_acc=0.6504 best=0.6504 improved=True
[arcface_m0p4_s64_seed4002] epoch 2/15 val_acc=0.6551 best=0.6551 improved=True
[arcface_m0p4_s64_seed4002] epoch 3/15 val_acc=0.6586 best=0.6586 improved=True
[arcface_m0p4_s64_seed4002] epoch 4/15 val_acc=0.6594 best=0.6594 improved=True
[arcface_m0p4_s64_seed4002] epoch 5/15 val_acc=0.6605 best=0.6605 improved=True
[arcface_m0p4_s64_seed4002] epoch 6/15 val_acc=0.6613 best=0.6613 improved=True
[arcface_m0p4_s64_seed4002] epoch 7/15 val_acc=0.6616 best=0.

In [19]:
REPOSITORY_ROOT = next(
    root for root in (Path.cwd().resolve(), Path.cwd().resolve().parent)
    if (root / "GLOSSARY.md").exists() and (root / "face_name_metric_learning").exists()
)

MANIFEST_PATH = REPOSITORY_ROOT / "data" / "processed" / "manifest.csv"
SOURCE_HDF5_PATH = REPOSITORY_ROOT / "data" / "processed" / "faces.h5"
PRETRAINED_CHECKPOINT_PATH = (
    REPOSITORY_ROOT / "models" / "pretrained" / "iresnet100.pth"
)
TRIAL_RESULTS_PATH = (
    REPOSITORY_ROOT / "artifacts" / "face_name_metric_learning" / "trial_results.csv"
)
EXPECTED_SPLIT_COUNTS = {"train": 45_078, "validation": 9_660, "test": 9_660}
EXPECTED_CLASS_COUNT = 100

HDF5_PATH = REPOSITORY_ROOT / "data" / "processed" / "faces.h5"

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

import importlib
import face_name_metric_learning.pytorch_training as pytorch_training

importlib.reload(pytorch_training)

trial_rows = pytorch_training.train_and_select_candidates(
    repository_root=REPOSITORY_ROOT,
    manifest_path=MANIFEST_PATH,
    hdf5_path=HDF5_PATH,
    trial_results_path=TRIAL_RESULTS_PATH,
    pretrained_checkpoint_path=PRETRAINED_CHECKPOINT_PATH,
    manifest_rows=manifest_rows,
    config=DEFAULT_TRAINING_CONFIG,
    max_new_trials=2,
)

[triplet_margin0p1_seed4002] already in ledger, skipping.
[triplet_margin0p2_seed4002] already in ledger, skipping.
[triplet_margin0p4_seed4002] already in ledger, skipping.
[arcface_m0p2_s30_seed4002] already in ledger, skipping.
[arcface_m0p2_s64_seed4002] already in ledger, skipping.
[arcface_m0p4_s30_seed4002] already in ledger, skipping.
[arcface_m0p4_s64_seed4002] already in ledger, skipping.
[supcon_t0p07_seed4002] already in ledger, skipping.
Loading /nfs/home/aqu4eb/DS4002_Project3/data/processed/faces.h5 into RAM (~9.7 GB, one-time)...
Loaded into RAM.
[supcon_t0p1_seed4002] epoch 1/15 val_acc=0.6471 best=0.6471 improved=True
[supcon_t0p1_seed4002] epoch 2/15 val_acc=0.6544 best=0.6544 improved=True
[supcon_t0p1_seed4002] epoch 3/15 val_acc=0.6542 best=0.6544 improved=False
[supcon_t0p1_seed4002] epoch 4/15 val_acc=0.6565 best=0.6565 improved=True
[supcon_t0p1_seed4002] epoch 5/15 val_acc=0.6594 best=0.6594 improved=True
[supcon_t0p1_seed4002] epoch 6/15 val_acc=0.6617 best=0

In [20]:
ledger = validate_trial_ledger(TRIAL_RESULTS_PATH)
assert len(ledger["rows"]) == 10
print("Selected trials:", ledger["selected_trials"], flush=True)

checkpoint_paths = [trial_checkpoint_path(REPOSITORY_ROOT, row) for row in ledger["rows"]]
assert all(path.is_file() for path in checkpoint_paths)
assert all("models/metric_learning" in path.as_posix() for path in checkpoint_paths)
print("Best checkpoints:", flush=True)
for path in checkpoint_paths:
    print(path.relative_to(REPOSITORY_ROOT), flush=True)

Selected trials: {'triplet_margin': 'triplet_margin0p4_seed4002', 'arcface': 'arcface_m0p2_s30_seed4002', 'supcon': 'supcon_t0p2_seed4002'}
Best checkpoints:
models/metric_learning/triplet_margin0p1_seed4002/best.pt
models/metric_learning/triplet_margin0p2_seed4002/best.pt
models/metric_learning/triplet_margin0p4_seed4002/best.pt
models/metric_learning/arcface_m0p2_s30_seed4002/best.pt
models/metric_learning/arcface_m0p2_s64_seed4002/best.pt
models/metric_learning/arcface_m0p4_s30_seed4002/best.pt
models/metric_learning/arcface_m0p4_s64_seed4002/best.pt
models/metric_learning/supcon_t0p07_seed4002/best.pt
models/metric_learning/supcon_t0p1_seed4002/best.pt
models/metric_learning/supcon_t0p2_seed4002/best.pt
